## STOCK PRICE PREDICTION WITH TRANSFORMERS

In [ ]:
class StockPredictionException(Exception):
    """
    Custom exception for the stock prediction project.
    Ensures that all errors raised within the codebase are traceable and standardized.
    """
    def __init__(self, message: str, error_detail: str = ""):
        self.message = message
        self.error_detail = error_detail
        super().__init__(self.message)

    def __str__(self):
        if self.error_detail:
            return f"{self.message}\nDetail: {self.error_detail}"
        return self.message

### Robust Data Acquisition

In [ ]:
import os
import pandas as pd
import kagglehub
from pathlib import Path
from tqdm import tqdm

# Configuration
RAW_DATA_DIR = Path("data/raw")
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

KAGGLE_DATASET = "borismarjanovic/price-volume-data-for-all-us-stocks-etfs"

# download dataset with kaggle api
print("Downloading dataset from Kaggle...")
dataset_path = kagglehub.dataset_download(KAGGLE_DATASET)
print(f"Dataset downloaded to: {dataset_path}")

# Inspect the downloaded folder structure and find the correct stocks folder
src = Path(dataset_path)

stocks_folder = src / "Stocks" #look for dir by the name: "Stocks"
if not stocks_folder.exists():
    # Fallback: try Data/Stocks folder
    stocks_folder = src / "Data" / "Stocks"
if not stocks_folder.exists():
    raise FileNotFoundError("Could not find 'Stocks' folder in dataset.")

# Find all text files (the dataset uses .txt extension for CSV data)
txt_files = list(stocks_folder.glob("*.txt")) #use .glob to iterdir()
print(f"Found {len(txt_files)} stock files.")

if not txt_files:
    raise FileNotFoundError("No .txt files found in Stocks folder. Check dataset structure.")

# Convert .txt (CSV) to Parquet (idempotent caching)
for txt_path in tqdm(txt_files, desc="Converting to Parquet"):
    ticker = txt_path.stem.upper()  # e.g., "AAPL.US" -> keep as is
    parquet_path = RAW_DATA_DIR / f"{ticker}.parquet"

    if parquet_path.exists():
        continue  # checks existence and proceeds

    try:
        # Read .txt as CSV (comma separated, with header, date index)
        df = pd.read_csv(txt_path, index_col=0, parse_dates=True)
        df.dropna(how='all', inplace=True)
        df.sort_index(inplace=True)
        df.to_parquet(parquet_path)
    except Exception as e:
        print(f"Skipping {ticker} due to error: {e}")

print(f"Data cached in {RAW_DATA_DIR.resolve()}")

### Extract Valid PArquets from Directory

In [ ]:
import pandas as pd
from pathlib import Path

RAW_DATA_DIR = Path("data/raw")
MIN_ROWS = 500 #min number of rows to be considered as a valid parquet
REQUIRED_COLS = ["Open", "High", "Low", "Close", "Volume"] #required columns each parquet must have

parquet_files = list(RAW_DATA_DIR.glob("*.parquet")) #search for the .parquet extension
print(f"Total parquet files before validation: {len(parquet_files)}")

valid_count = 0
removed_count = 0
for pq_path in parquet_files:
    ticker = pq_path.stem #get the name of the parquet
    try:
        df = pd.read_parquet(pq_path)
        # Check minimum rows and required columns
        if df.shape[0] < MIN_ROWS or not all(col in df.columns for col in REQUIRED_COLS):
            pq_path.unlink()
            removed_count += 1
            continue
        # Check for empty dataframe and remove from the selected parquets
        if df.dropna(how='all').empty:
            pq_path.unlink()
            removed_count += 1
            continue
        valid_count += 1
    except Exception:
        # Corrupt parquet – delete it
        pq_path.unlink()
        removed_count += 1

print(f"Valid stocks: {valid_count}")
print(f"Removed: {removed_count}")

### Extract Key ETFs (SPY, QQQ, IWM) for Market‑Aware Features

In [ ]:
#run the extract etf again and choose a few parquets to see if they are cached
from pathlib import Path
import pandas as pd
from tqdm import tqdm

RAW_DATA_DIR = Path("data/raw")
DATASET_ROOT = Path("/kaggle/input/price-volume-data-for-all-us-stocks-etfs")  # adjust if different

etfs_folder = DATASET_ROOT / "ETFs"
if not etfs_folder.exists():
    etfs_folder = DATASET_ROOT / "Data" / "ETFs" #fallback on the dir structure dataset_root/data/etfs

target_etfs = ["SPY.US", "QQQ.US", "IWM.US"] #create a short list of etfs-big 3 macro benchmarks

for ticker in target_etfs:
    src_path = etfs_folder / f"{ticker.lower()}.txt"  # file names are lowercase
    if not src_path.exists():
        print(f"{ticker} not found in ETFs folder. Skipping.")
        continue
    dst_path = RAW_DATA_DIR / f"{ticker}.parquet"
    if dst_path.exists():
        print(f"{ticker} already cached.")
        continue
    try:
        df = pd.read_csv(src_path, index_col=0, parse_dates=True)
        df.dropna(how='all', inplace=True)
        df.sort_index(inplace=True)
        df.to_parquet(dst_path)
        print(f"{ticker} converted and cached.")
    except Exception as e:
        print(f"Error processing {ticker}: {e}")

In [ ]:
#confirmation of the conveeted and cached etfs
from pathlib import Path
import pandas as pd

RAW_DATA_DIR = Path("data/raw")

# The original dataset path (adjust if Colab location differs)
# Correct DATASET_ROOT to the actual download path from kagglehub
DATASET_ROOT = Path("/root/.cache/kagglehub/datasets/borismarjanovic/price-volume-data-for-all-us-stocks-etfs/versions/3")

# Locate the ETFs folder
etfs_folder = DATASET_ROOT / "ETFs"
if not etfs_folder.exists():
    etfs_folder = DATASET_ROOT / "Data" / "ETFs"

target_etfs = ["SPY.US", "QQQ.US", "IWM.US"]

for ticker in target_etfs:
    # Dataset uses lowercase filenames
    src_path = etfs_folder / f"{ticker.lower()}.txt"
    if not src_path.exists():
        print(f"{ticker} not found in ETFs folder. Skipping.")
        continue

    dst_path = RAW_DATA_DIR / f"{ticker}.parquet"
    if dst_path.exists():
        print(f"{ticker} already cached.")
        continue

    try:
        df = pd.read_csv(src_path, index_col=0, parse_dates=True)
        df.dropna(how='all', inplace=True)
        df.sort_index(inplace=True)
        df.to_parquet(dst_path)
        print(f"{ticker} converted and cached.")
    except Exception as e:
        print(f"Error processing {ticker}: {e}")

In [ ]:
#load one etf as a dataframe and display 10 rows
import pandas as pd
from pathlib import Path

RAW_DATA_DIR = Path("data/raw")
etf_path = RAW_DATA_DIR / "SPY.US.parquet"
df = pd.read_parquet(etf_path)
df.head(10)

### Exploratory Data Analysis (with SPY)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Settings
sns.set_style("dark")
plt.rcParams["figure.figsize"] = (14, 7)

RAW_DATA_DIR = Path("data/raw")
# sample tickers for analysis
SAMPLE_TICKERS = ["AAPL.US", "MSFT.US", "TSLA.US", "QQQ.US"]

# Load sample data
sample_dfs = {}
for ticker in SAMPLE_TICKERS:
    file_path = RAW_DATA_DIR / f"{ticker}.parquet"
    if file_path.exists():
        df = pd.read_parquet(file_path)
        sample_dfs[ticker] = df
        print(f"{ticker}: {df.shape[0]} rows, from {df.index[0].date()} to {df.index[-1].date()}") #print the first date and last date
    else:
        print(f"{ticker}: file not found – check ticker naming")


# Price and Volume Plots
fig, axes = plt.subplots(len(sample_dfs), 2, figsize=(16, 3 * len(sample_dfs)))
if len(sample_dfs) == 1:
    axes = [axes]

for i, (ticker, df) in enumerate(sample_dfs.items()):
    ax1 = axes[i][0] if len(sample_dfs) > 1 else axes[0]
    ax2 = axes[i][1] if len(sample_dfs) > 1 else axes[1]

    # Close price
    ax1.plot(df.index, df["Close"], color="blue", lw=1)
    ax1.set_title(f"{ticker} Close Price")
    ax1.set_ylabel("Price ($)")

    # Volume
    ax2.bar(df.index, df["Volume"], color="black",alpha=0.2, width=3)
    ax2.set_title(f"{ticker} Volume")
    ax2.set_ylabel("Shares")

    ax1.tick_params(axis='x', rotation=45)
    ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


# Returns and Realised Volatility
for ticker, df in sample_dfs.items():
    df["Log_Ret"] = np.log(df["Close"] / df["Close"].shift(1))
    df["Realised_Vol_21"] = df["Log_Ret"].rolling(21).std() * np.sqrt(252)  # Annualised

    fig, ax = plt.subplots(2, 1, figsize=(14, 6))
    ax[0].plot(df.index, df["Log_Ret"], lw=0.5, color="green")
    ax[0].set_title(f"{ticker} Daily Log Returns")
    ax[0].set_ylabel("Log Return")

    ax[1].plot(df.index, df["Realised_Vol_21"], lw=1, color="red")
    ax[1].set_title(f"{ticker} 21-Day Rolling Annualised Volatility")
    ax[1].set_ylabel("Volatility")
    ax[1].axhline(df["Realised_Vol_21"].median(), color="black", linestyle="--", label="Median")
    ax[1].legend()

    plt.tight_layout()
    plt.show()


# Missing Values Analysis
for ticker, df in sample_dfs.items():
    missing = df.isnull().sum()
    missing_pct = (missing / len(df)) * 100
    missing_table = pd.DataFrame({"Missing Count": missing, "Percentage": missing_pct})
    missing_table = missing_table[missing_table["Missing Count"] > 0]
    if not missing_table.empty:
        print(f"\n{ticker} Missing Values:\n{missing_table}")
    else:
        print(f"\n{ticker}: No missing values.")


# Cross‑correlation with QQQ (systematic risk)
if "QQQ.US" in sample_dfs and len(sample_dfs) > 1:
    qqq_ret = sample_dfs["QQQ.US"]["Close"].pct_change().dropna()

    for ticker, df in sample_dfs.items():
        if ticker == "QQQ.US":
            continue
        stock_ret = df["Close"].pct_change().dropna()
        common_idx = qqq_ret.index.intersection(stock_ret.index)
        corr = stock_ret.loc[common_idx].corr(qqq_ret.loc[common_idx])
        print(f"Correlation of {ticker} daily returns with SPY: {corr:.3f}")

In [ ]:
sample_dfs

### Feature Engineering (Volatility‑Informed)

This is where the important stuff for the dataset is done

In [ ]:
!pip install arch -q

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
from arch import arch_model
import warnings
warnings.filterwarnings("ignore")

#defining helper functions to support the project
def compute_rsi(series: pd.Series, period: int = 14) -> pd.Series:
    """Relative Strength Index."""
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.rolling(period).mean()
    avg_loss = loss.rolling(period).mean()
    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

def compute_macd(series: pd.Series, fast=12, slow=26, signal=9):
    """MACD line, signal line, histogram."""
    ema_fast = series.ewm(span=fast, adjust=False).mean()
    ema_slow = series.ewm(span=slow, adjust=False).mean()
    macd_line = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=signal, adjust=False).mean()
    histogram = macd_line - signal_line
    return macd_line, signal_line, histogram

def compute_atr(df: pd.DataFrame, period: int = 14) -> pd.Series:
    """Computes the average true range which measures how big the stock's daily
    moves are, including the gaps
    """

    high = df["High"]
    low = df["Low"]
    close = df["Close"]
    tr1 = high - low
    tr2 = (high - close.shift()).abs()
    tr3 = (low - close.shift()).abs()
    true_range = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    return true_range.rolling(period).mean()

def compute_bollinger_band_width(series: pd.Series, period=20, num_std=2):
    """This takes a stocks features and measures how wide the bollinger bands are"""

    sma = series.rolling(period).mean()
    std = series.rolling(period).std()
    # Handle cases where sma might be zero or very close to zero to avoid inf/NaN
    with np.errstate(divide='ignore', invalid='ignore'):
        bb_width = ( (sma + num_std*std) - (sma - num_std*std) ) / sma
        bb_width = bb_width.replace([np.inf, -np.inf], np.nan)
    return bb_width # relative width

def compute_parkinson_volatility(df: pd.DataFrame, period: int = 21) -> pd.Series:
    """It uses the high and low prices of the day for a stock to measure volatility"""

    # High-Low range volatility
    hl = np.log(df["High"] / df["Low"])
    return np.sqrt( (hl**2).rolling(period).sum() / (4 * np.log(2) * period) )


def compute_stochastic(df: pd.DataFrame, period: int = 14) -> (pd.Series, pd.Series):
    """
    Stochastic Oscillator %K and %D.
    %K = 100 * (Close - Low_min) / (High_max - Low_min)
    %D = 3‑day SMA of %K
    """
    low_min = df["Low"].rolling(period).min()
    high_max = df["High"].rolling(period).max()
    pct_k = 100 * (df["Close"] - low_min) / (high_max - low_min + 1e-8)
    pct_d = pct_k.rolling(3).mean()
    return pct_k, pct_d

def compute_roc(series: pd.Series, period: int = 10) -> pd.Series:
    """Rate of Change: (price / price(period ago) - 1) * 100"""
    return (series / series.shift(period) - 1) * 100

def compute_obv(df: pd.DataFrame) -> pd.Series:
    """On‑Balance Volume: cumulative signed volume."""
    return (np.sign(df["Close"].diff()) * df["Volume"]).fillna(0).cumsum()

def fit_garch_proxy(returns: pd.Series, alpha: float = 0.06) -> pd.Series:
    """EWMA fallback volatility."""
    return returns.ewm(alpha=alpha, adjust=False).std() * np.sqrt(252)

def fit_garch(returns: pd.Series, fallback_alpha: float = 0.06) -> pd.Series:
    """
    Fit GARCH(1,1) on log‑return series using an expanding window.
    Falls back to EWMA if fitting fails or insufficient data.
    """
    clean = returns.dropna()
    if len(clean) < 500:
        return fit_garch_proxy(returns, fallback_alpha)

    vol = pd.Series(np.nan, index=returns.index)
    min_window = 500
    refit_freq = 252  # re‑estimate every ~1 year

    for end_idx in range(min_window, len(clean) + 1, refit_freq):
        train = clean.iloc[:end_idx]
        try:
            model = arch_model(train, mean='Zero', vol='GARCH', p=1, q=1, dist='normal')
            res = model.fit(disp='off', show_warning=False)
            forecast_horizon = min(refit_freq, len(clean) - end_idx)
            forecast = res.forecast(horizon=forecast_horizon, reindex=False)
            cond_vol = np.sqrt(forecast.variance.values[-1, :])
            forecast_start = clean.index[end_idx]
            forecast_end = clean.index[end_idx + forecast_horizon - 1]
            vol.loc[forecast_start:forecast_end] = cond_vol[:forecast_horizon]
        except Exception as e:
            print(f"GARCH fitting failed: {e}. Using EWMA fallback.")
            segment = clean.iloc[end_idx:end_idx + refit_freq]
            if len(segment) > 0:
                vol.loc[segment.index] = fit_garch_proxy(returns.loc[segment.index], fallback_alpha)

    # Fill remaining NaN (first min_window days) with EWMA
    nan_mask = vol.isna()
    if nan_mask.any():
        vol.loc[nan_mask] = fit_garch_proxy(returns.loc[nan_mask], fallback_alpha)

    return vol * np.sqrt(252)  # annualise

#setup dummy configs

cfg = {
    "use_garch":True,
    "rsi_periods":[14],
    "vol_windows": [5, 21, 63],
    "macd_params":{"standard": [12, 26, 9]},
    "atr_period":14,
    "bb_periods": [20],
    "parkinson_period": 21,
    "stoch_period": 21,
    "beta_window": 63,
    "direction_adaptive_factor": 0.7
}

def engineer_features(
    df: pd.DataFrame,spy_df: pd.DataFrame = None,vix_df: pd.DataFrame = None,cfg_features: dict = None
) -> pd.DataFrame:
    """
    Build the full feature set from raw OHLCV data.

    Parameters
    ----------
    df : pd.DataFrame
        Raw OHLCV data for a single stock (index=DateTime).
    spy_df : pd.DataFrame, optional
        SPY data (must contain 'Close') for beta computation.
    vix_df : pd.DataFrame, optional
        VIX data (must contain 'VIX') to be merged.
    cfg_features : dict
        Feature parameters from config.yaml.

    Returns
    -------
    pd.DataFrame with all feature and target columns.
    """
    if cfg_features is None:
        cfg_features = {}

    data = df.copy()

    # Basic returns
    data["Return_1d"] = data["Close"].pct_change() #1d day return prices
    data["Log_Ret_1d"] = np.log(data["Close"] / data["Close"].shift(1)) #log of the 1d return
    data["Return_5d"] = data["Close"].pct_change(5) #5 day return
    data["Volume_Change"] = data["Volume"].pct_change()
    data["High_Low_Pct"] = (data["High"] - data["Low"]) / data["Close"]
    data["Close_Open_Pct"] = (data["Close"] - data["Open"]) / data["Open"]

    # Realised volatility
    vol_windows = cfg_features.get("vol_windows", [5, 21, 63])
    for win in vol_windows:
        data[f"Real_Vol_{win}d"] = data["Log_Ret_1d"].rolling(win).std() * np.sqrt(252)

    # GARCH(1,1)
    if cfg_features.get("use_garch", True):
        data["GARCH_vol"] = fit_garch(data["Log_Ret_1d"],
                                      fallback_alpha=cfg_features.get("garch_ewma_alpha", 0.06))
    else:
        data["GARCH_vol"] = fit_garch_proxy(data["Log_Ret_1d"],
                                            alpha=cfg_features.get("garch_ewma_alpha", 0.06))

    # RSI (multiple periods)
    rsi_periods = cfg_features.get("rsi_periods", [14])
    for per in rsi_periods:
        data[f"RSI_{per}"] = compute_rsi(data["Close"], per)

    # MACD variants
    macd_params = cfg_features.get("macd_params", {"standard": [12, 26, 9]})
    for name, (f, s, sig) in macd_params.items():
        macd_line, signal_line, hist = compute_macd(data["Close"], fast=f, slow=s, signal=sig)
        data[f"MACD_{name}"] = macd_line
        data[f"MACD_signal_{name}"] = signal_line
        data[f"MACD_hist_{name}"] = hist

    # ---------- ATR ----------
    data["ATR_14"] = compute_atr(data, cfg_features.get("atr_period", 14))

    # Bollinger Band width (multiple periods)
    bb_periods = cfg_features.get("bb_periods", [20])
    for per in bb_periods:
        data[f"BB_width_{per}"] = compute_bollinger_band_width(data["Close"], period=per)

    # Parkinson volatility
    data["Parkinson_vol_21"] = compute_parkinson_volatility(data, cfg_features.get("parkinson_period", 21))

    # Stochastic Oscillator
    stoch_period = cfg_features.get("stoch_period", None)
    if stoch_period:
        stoch_k, stoch_d = compute_stochastic(data, stoch_period)
        data["Stoch_%K"] = stoch_k
        data["Stoch_%D"] = stoch_d

    # Rate of Change (ROC)
    data["ROC_10"] = compute_roc(data["Close"], 10)

    # On‑Balance Volume (OBV)
    data["OBV"] = compute_obv(data)

    #  Moving averages & crosses
    data["SMA_20"] = data["Close"].rolling(20).mean()
    data["SMA_50"] = data["Close"].rolling(50).mean()
    data["SMA_200"] = data["Close"].rolling(200).mean()
    data["SMA_20_50_cross"] = (data["SMA_20"] - data["SMA_50"]) / data["Close"]
    data["SMA_50_200_cross"] = (data["SMA_50"] - data["SMA_200"]) / data["Close"]

    #  Temporal features
    data["DayOfWeek"] = data.index.dayofweek
    data["Month"] = data.index.month
    data["DayOfWeek_sin"] = np.sin(2 * np.pi * data["DayOfWeek"] / 5)
    data["DayOfWeek_cos"] = np.cos(2 * np.pi * data["DayOfWeek"] / 5)

    # Systematic risk: rolling beta to SPY
    if spy_df is not None:
        spy_ret = spy_df["Close"].pct_change()
        combined = pd.concat([data["Return_1d"], spy_ret], axis=1)
        combined.columns = ["stock_ret", "spy_ret"]
        combined.dropna(inplace=True)
        beta_window = cfg_features.get("beta_window", 63)
        rolling_cov = combined["stock_ret"].rolling(beta_window).cov(combined["spy_ret"])
        rolling_var = combined["spy_ret"].rolling(beta_window).var()
        beta = rolling_cov / rolling_var
        data["Beta_63"] = beta
    else:
        data["Beta_63"] = np.nan

    # VIX
    if vix_df is not None:
        data = data.join(vix_df, how="left")
    else:
        data["VIX"] = np.nan

    # Target columns
    data["Target_Log_Ret_Next"] = data["Log_Ret_1d"].shift(-1)

    # Adaptive direction threshold
    adaptive_factor = cfg_features.get("direction_adaptive_factor", 0.7)
    threshold = adaptive_factor * data["Real_Vol_21d"]  # 21‑day realised vol must exist
    next_ret = data["Return_1d"].shift(-1)
    data["Target_Direction"] = np.where(
        next_ret > threshold, 2,
        np.where(next_ret < -threshold, 0, 1)
    )

    data["Target_Vol_Next"] = data["Real_Vol_21d"].shift(-1)

    #  Drop rows with NaN
    data.dropna(inplace=True)
    return data


In [ ]:
import yfinance as yf
import warnings
import pandas as pd
from pathlib import Path
RAW_DATA_DIR = Path("data/raw")
VIX_TICKER = "VIX"
VIX_PATH = RAW_DATA_DIR / f"{VIX_TICKER}.parquet"

vix_df = None

if VIX_PATH.exists():
    print(f"Loading VIX data from {VIX_PATH}...")
    vix_df = pd.read_parquet(VIX_PATH)
else:
    print(f"Downloading VIX data ({VIX_TICKER}) from Yahoo Finance...")
    try:
        # Download VIX data
        vix_data = yf.download(VIX_TICKER, start="1990-01-01", end="2024-01-01", progress=False)
        if not vix_data.empty:
            # Rename 'Adj Close' to 'Close' for consistency, and select relevant columns
            vix_data = vix_data[['Open', 'High', 'Low', 'Close', 'Volume']]
            # The engineer_features expects a 'VIX' column specifically, so we'll add it if 'Close' is VIX.
            vix_data.rename(columns={'Close': 'VIX'}, inplace=True)
            vix_df = vix_data.sort_index()
            vix_df.to_parquet(VIX_PATH)
            print(f"VIX data downloaded and saved to {VIX_PATH}")
        else:
            print("Failed to download VIX data or data is empty.")
    except Exception as e:
        warnings.warn(f"Could not download VIX data: {e}. VIX feature will be NaN.")

if vix_df is not None:
    vix_df.index = pd.to_datetime(vix_df.index)
    # Ensure the VIX column is present if using it as a feature
    if 'VIX' not in vix_df.columns:
        if 'Close' in vix_df.columns: # Fallback if not renamed properly earlier
            vix_df['VIX'] = vix_df['Close']
        else:
            warnings.warn("VIX DataFrame does not contain a 'VIX' or 'Close' column. VIX feature might be NaN.")


# The rest of the original code follows a similar structure, ensure it works with the new vix_df variable.
# Example: features_sample = engineer_features(df_sample, spy_df, vix_df)

In [ ]:
#lets see whats in the vix parquet
RAW_DATA_DIR = Path("data/raw")
VIX_TICKER = "VIX"
VIX_PATH = RAW_DATA_DIR / f"{VIX_TICKER}.parquet"

#use pd.read_parquet
vix_df = pd.read_parquet(VIX_PATH)
vix_df.head()

In [ ]:
#apply the feature engineering on a sample parquet

sample_ticker = "AAPL.US"
df_sample = pd.read_parquet(Path("data/raw") / f"{sample_ticker}.parquet")
spy_df = pd.read_parquet(Path("data/raw") / "SPY.US.parquet") if Path("data/raw/SPY.US.parquet").exists() else None

features_sample = engineer_features(df_sample, spy_df)
print(f"Feature shape: {features_sample.shape}")
print(f"Feature columns: {list(features_sample.columns)}")

# Check for inf/nan after engineering
nan_count = features_sample.isnull().sum().sum()
inf_count = (features_sample.abs() == np.inf).sum().sum()
if nan_count > 0:
    print("WARNING: NaNs still present after feature engineering. Inspect data.")
    print(features_sample.isnull().sum()[features_sample.isnull().sum() > 0])
elif inf_count > 0:
    print("WARNING: Infs still present after feature engineering. Inspect data.")
    print((features_sample.abs() == np.inf).sum()[(features_sample.abs() == np.inf).sum() > 0])
else:
    print("No NaNs or Infs found in features_sample.")

print("\nMax values of features (excluding targets and raw OHLCV):")
# Exclude raw OHLCV and target columns for a cleaner view of engineered features
exclude_cols = ['Open', 'High', 'Low', 'Close', 'Volume', 'OpenInt',
                'Target_Close_Next', 'Target_Direction', 'Target_Vol_Next', 'Target_Log_Ret_Next']
engineered_feature_cols = [c for c in features_sample.columns if c not in exclude_cols]
print(features_sample[engineered_feature_cols].max().sort_values(ascending=False).head(10))
print("\nMin values of features (excluding targets and raw OHLCV):")
print(features_sample[engineered_feature_cols].min().sort_values(ascending=True).head(10))

features_sample[["Close", "Return_1d", "Real_Vol_21d", "GARCH_vol_21", "RSI_14", "ATR_14", "Beta_63"]].tail(10)


In [ ]:
features_sample['Target_Log_Ret_Next'].head(10)

### PyTorch Dataset Classes (Pre‑training & Fine‑tuning)

In [ ]:
import torch
from torch.utils.data import Dataset
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Dict, List, Optional
from sklearn.preprocessing import StandardScaler # Import StandardScaler

class PreTrainingDataset(Dataset):
    """
    Dataset for pre‑training on multiple stocks.
    Each sample is a fixed‑length window of feature vectors from a random stock,
    with the corresponding shifted close prices as targets.
    Only the LAST target is used for loss (next‑day prediction), but we keep
    the full target sequence for completeness.
    """
    def __init__(self, data_dir="data/processed", tickers=None, seq_len=60,
                 feature_cols=None, target_col="Target_Log_Ret_Next"):
        self.data_dir = Path(data_dir)
        self.seq_len = seq_len
        self.target_col = target_col
        self.scaler = None # Initialize scaler

        # Find all parquet files
        if tickers is None:
            self.ticker_files = list(self.data_dir.glob("*.parquet"))
        else:
            self.ticker_files = [self.data_dir / f"{t}.parquet" for t in tickers]

        self.data_raw = {}   # maps ticker -> raw df
        self.target_idx = None
        self.feature_cols = feature_cols # Store feature_cols here temporarily
        self.feature_indices = None

        all_features_data = [] # To collect features for scaler fitting

        # First pass: Load data, determine feature columns, collect features for scaling
        for f in self.ticker_files:
            if f.exists():
                df = pd.read_parquet(f)

                # ADDITIONAL SAFEGUARD: Ensure no NaNs or Infs persist after loading
                df.replace([np.inf, -np.inf], np.nan, inplace=True)
                df.dropna(inplace=True)
                if df.empty:
                    # print(f"WARNING: {f.stem} became empty after NaN/Inf removal. Skipping.")
                    continue

                if self.feature_cols is None:
                    # Determine feature columns from the first loaded df
                    all_cols = df.columns.tolist()
                    if target_col not in all_cols:
                        raise ValueError(f"Target column '{target_col}' not found in processed data.")
                    # Feature columns: all numeric columns except the target
                    self.feature_cols = [c for c in all_cols if c != target_col]
                    # Also find target_idx if not already set (should be from first df)
                    self.target_idx = all_cols.index(target_col)

                # Collect features for scaler fitting
                all_features_data.append(df[self.feature_cols].values)
                self.data_raw[f.stem] = df # Store raw df for target extraction later

        if not self.data_raw:
            raise ValueError("No ticker files found or loaded with sufficient data for feature engineering.")

        # Fit StandardScaler
        combined_features = np.vstack(all_features_data)

        # Explicitly handle potential NaNs or Infs before fitting scaler
        # Filter out rows that contain any non-finite values
        finite_rows = np.all(np.isfinite(combined_features), axis=1)
        if not np.all(finite_rows):
            print(f"WARNING: {np.sum(~finite_rows)} rows containing non-finite values (NaN or Inf) detected in combined features for scaler fitting. These rows will be removed.")
            combined_features = combined_features[finite_rows]
            if combined_features.shape[0] == 0:
                raise ValueError("No finite data left after cleaning for StandardScaler fitting.")

        # Debugging: check values before scaling
        print(f"DEBUG: Shape of combined_features before scaling: {combined_features.shape}")
        if combined_features.size > 0: # Check if array is not empty before min/max
            print(f"DEBUG: Min value in combined_features before scaling: {np.min(combined_features)}")
            print(f"DEBUG: Max value in combined_features before scaling: {np.max(combined_features)}")
            print(f"DEBUG: Contains NaN before scaling: {np.any(np.isnan(combined_features))}")
            print(f"DEBUG: Contains Inf before scaling: {np.any(np.isinf(combined_features))}")
            # New check for extremely large but finite numbers
            max_float64 = np.finfo(np.float64).max
            if np.max(combined_features) >= max_float64 * 0.9: # Check if very close to max
                 print(f"WARNING: Combined features contain values extremely close to float64 max: {np.max(combined_features)}")
            if np.min(combined_features) <= -max_float64 * 0.9: # Check if very close to min
                 print(f"WARNING: Combined features contain values extremely close to float64 min: {np.min(combined_features)}")
        else:
            print("DEBUG: combined_features is empty after initial filtering.")


        self.scaler = StandardScaler()
        self.scaler.fit(combined_features)
        print(f"Fitted StandardScaler on {combined_features.shape[0]} samples with {combined_features.shape[1]} features.")
        del combined_features, all_features_data # Free memory

        self.data_scaled = {} # Store scaled features and targets
        total_days = 0
        # Second pass: Scale features and prepare data for __getitem__
        for f_stem, df_raw in self.data_raw.items():
            scaled_features = self.scaler.transform(df_raw[self.feature_cols].values)
            target_values = df_raw[self.target_col].values

            # Reconstruct array: scaled features then target
            temp_array = np.zeros((df_raw.shape[0], len(self.feature_cols) + 1), dtype=np.float32)
            temp_array[:, :len(self.feature_cols)] = scaled_features
            temp_array[:, len(self.feature_cols)] = target_values

            self.data_scaled[f_stem] = temp_array
            total_days += len(df_raw)

        # Update target_idx and feature_indices for the new structure (features then target)
        self.feature_indices = list(range(len(self.feature_cols)))
        self.target_idx = len(self.feature_cols) # Target is now at the end of the feature block

        print(f"Loaded {len(self.data_scaled)} tickers, {total_days} total days. Feature dim: {len(self.feature_cols)}")

        # Build index map
        self.index_map = []
        for ticker, arr in self.data_scaled.items():
            if len(arr) > seq_len:
                self.index_map.append((ticker, len(arr) - seq_len))
        if not self.index_map:
            raise ValueError("No tickers with sufficient data (after scaling).")

    def __len__(self):
        return sum(m for _, m in self.index_map)

    def __getitem__(self, idx):
        # Find ticker and start position
        cum = 0
        for t, m in self.index_map:
            if idx < cum + m:
                ticker = t
                start = idx - cum
                break
            cum += m
        else:
            # Fallback for random access if idx exceeds precomputed map (should not happen with proper __len__)
            ticker, m = self.index_map[np.random.randint(len(self.index_map))]
            start = np.random.randint(0, m)

        arr = self.data_scaled[ticker]
        # Input window: scaled features only
        x = arr[start:start + self.seq_len, self.feature_indices]
        # Target: next day's log return (the target column at position start+seq_len, which is self.target_idx)
        y = arr[start + self.seq_len, self.target_idx]   # scalar
        mask = torch.ones(1, dtype=torch.bool)           # we only care about this single target

        return (
            torch.from_numpy(x).float(),
            torch.tensor(y).float(),   # log return
            mask
        )

    def get_feature_dim(self):
        return len(self.feature_cols)

In [ ]:
import torch
from torch.utils.data import Dataset
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Dict, List, Optional
from sklearn.preprocessing import StandardScaler

class FineTuneDataset(Dataset):
    """
    Dataset for fine-tuning on a single stock.
    Returns: input features, price target, direction target, volatility target.
    """
    def __init__(
        self,
        df: pd.DataFrame,              # Single stock DataFrame with ALL feature columns and target columns
        seq_len: int = 60,
        feature_cols: Optional[List[str]] = None,
        target_price_col: str = "Target_Log_Ret_Next",
        target_direction_col: str = "Target_Direction",
        target_vol_col: str = "Target_Vol_Next",
        scaler: Optional[StandardScaler] = None # Optional pre-fitted scaler
    ):
        self.seq_len = seq_len
        self.scaler = scaler

        # Separate features and targets
        if feature_cols is None:
            exclude = [target_price_col, target_direction_col, target_vol_col, "Close"]  # exclude raw targets and original close
            self.feature_cols = [c for c in df.columns if c not in exclude]
        else:
            self.feature_cols = feature_cols

        features_raw = df[self.feature_cols].values.astype(np.float32)

        if self.scaler is None:
            # If no scaler provided, fit a new one (assumes this is for training data)
            self.scaler = StandardScaler()
            self.features = self.scaler.fit_transform(features_raw)
            print(f"Fitted StandardScaler on {len(df)} samples for fine-tuning.")
        else:
            # Use provided scaler to transform data (for validation/test data)
            self.features = self.scaler.transform(features_raw)

        self.price_target = df[target_price_col].values.astype(np.float32)
        self.dir_target = df[target_direction_col].values.astype(np.int64)
        self.vol_target = df[target_vol_col].values.astype(np.float32)

        # Number of valid starting indices
        self.valid_length = len(df) - seq_len

    def __len__(self):
        return self.valid_length

    def __getitem__(self, idx):
        x = self.features[idx:idx + self.seq_len]
        # The targets are for the day AFTER the last input day
        price_y = self.price_target[idx + self.seq_len]        # scalar
        dir_y = self.dir_target[idx + self.seq_len]            # scalar
        vol_y = self.vol_target[idx + self.seq_len]            # scalar
        return (
            torch.from_numpy(x).float(),
            torch.tensor(price_y).float(),
            torch.tensor(dir_y).long(),
            torch.tensor(vol_y).float()
        )

    def get_feature_dim(self) -> int:
        return self.features.shape[1]


### Dataset Preview (Single Stock Sample)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import StandardScaler

# Use the engineered features from Cell 4 (features_sample for AAPL.US)
sample = features_sample.copy()

# Identify columns that are NOT target columns (we'll use as features)
target_cols = ["Target_Close_Next", "Target_Direction", "Target_Vol_Next", "Close"]  # Close is raw, not used as feature
feature_cols = [c for c in sample.columns if c not in target_cols]

# Instantiate dataset with a short sequence length for easy viewing
seq_len = 5  # small window so we can print it

# First, create a dummy scaler to pass to the dataset for preview purposes
# In a real scenario, this would be the scaler fitted on the training data
dummy_scaler = StandardScaler()
dummy_scaler.fit(sample[feature_cols].values)

dataset = FineTuneDataset(
    df=sample,
    seq_len=seq_len,
    feature_cols=feature_cols,
    target_price_col="Target_Log_Ret_Next", # Changed to Target_Log_Ret_Next for consistency
    target_direction_col="Target_Direction",
    target_vol_col="Target_Vol_Next",
    scaler=dummy_scaler # Pass the dummy scaler
)

print(f"Total samples: {len(dataset)}")
print(f"Feature dimension: {dataset.get_feature_dim()}")

# Retrieve a few samples and display
samples_to_show = 10
rows = []
for i in range(samples_to_show):
    x, price, direction, vol = dataset[i]
    # x shape: (seq_len, num_features)
    # Convert input features to DataFrame for display (show first 5 features)
    x_df = pd.DataFrame(
        x.numpy(),
        columns=feature_cols,
        index=[f"Day-{seq_len-j}" for j in range(seq_len, 0, -1)]
    )
    # Build a row with the target values
    row = {
        "Sample Index": i,
        "Input Window (Days)": f"{seq_len} days",
        "Feature Matrix (first 3 cols)": x_df.iloc[:, :3].to_string(),
        "Price Target (next close)": f"{price.item():.2f}",
        "Direction Target (0=Down,1=Flat,2=Up)": direction.item(),
        "Volatility Target (next day)": f"{vol.item():.4f}"
    }
    rows.append(row)

display_df = pd.DataFrame(rows)
print("\n--- Dataset Output Preview ---")
display_df


### Feature Pipeline for All Stocks

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm

RAW_DATA_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Load SPY for systematic risk (rolling beta)
spy_path = RAW_DATA_DIR / "SPY.US.parquet"
spy_df = pd.read_parquet(spy_path) if spy_path.exists() else None

# Get all stock parquet files (exclude ETFs we don't want to process)
stock_files = [f for f in RAW_DATA_DIR.glob("*.parquet")
               if f.stem not in ["SPY.US", "QQQ.US", "IWM.US"]]  # we can process all stocks

print(f"Processing {len(stock_files)} stock files...")

for pq_path in tqdm(stock_files, desc="Engineering features"):
    ticker = pq_path.stem
    out_path = PROCESSED_DIR / f"{ticker}.parquet"
    if out_path.exists():
        continue  # idempotent

    try:
        df = pd.read_parquet(pq_path)
        # Ensure required columns
        required = ["Open", "High", "Low", "Close", "Volume"]
        if not all(c in df.columns for c in required):
            print(f"{ticker}: missing OHLCV columns, skipping.")
            continue

        # Engineer features
        feat_df = engineer_features(df, spy_df)
        feat_df.to_parquet(out_path)
    except Exception as e:
        print(f"{ticker}: error - {e}")

print(f"Processed features saved to {PROCESSED_DIR.resolve()}")


### Model Architecture

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
# -------------------------------
# 1. Core Transformer Components
# -------------------------------
class MultiHeadCausalAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        B, T, C = x.shape
        # Linear projections and reshape for multi-head
        q = self.W_q(x).view(B, T, self.n_heads, self.d_k).transpose(1,2)  # (B, n_heads, T, d_k)
        k = self.W_k(x).view(B, T, self.n_heads, self.d_k).transpose(1,2)
        v = self.W_v(x).view(B, T, self.n_heads, self.d_k).transpose(1,2)

        attn_scores = (q @ k.transpose(-2,-1)) / math.sqrt(self.d_k)
        if mask is not None:
            # mask shape: (T,T) or (B,1,T,T); broadcast
            attn_scores = attn_scores.masked_fill(mask == 0, float('-inf'))
        attn_probs = F.softmax(attn_scores, dim=-1)
        attn_probs = self.dropout(attn_probs)

        out = attn_probs @ v  # (B, n_heads, T, d_k)
        out = out.transpose(1,2).contiguous().view(B, T, C)
        out = self.W_o(out)
        return out


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.linear2(self.dropout(F.gelu(self.linear1(x))))

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadCausalAttention(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # Pre-LayerNorm
        x = x + self.dropout(self.attn(self.ln1(x), mask))
        x = x + self.dropout(self.ff(self.ln2(x)))
        return x

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class StockTransformer(nn.Module):
    def __init__(self, num_features, d_model=256, n_heads=8, n_layers=6,
                 d_ff=512, max_seq_len=252, dropout=0.1):
        """
        num_features: number of input features (dimensionality of each day's vector)
        d_model: embedding dimension
        max_seq_len: maximum number of days in a sequence (for positional embeddings)
        """
        super().__init__()
        self.d_model = d_model
        self.input_proj = nn.Linear(num_features, d_model)
        self.pos_embed = nn.Embedding(max_seq_len, d_model)

        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])
        self.final_ln = nn.LayerNorm(d_model)

        # ---- Three Output Heads ----
        # Price head: outputs mu and log_var (for Gaussian NLL)
        self.price_mu = nn.Linear(d_model, 1)
        self.price_logvar = nn.Linear(d_model, 1)

        # Direction head: 3 classes (Down, Flat, Up)
        self.direction_head = nn.Linear(d_model, 3)

        # Volatility head: single real value (realized volatility)
        self.volatility_head = nn.Linear(d_model, 1)

        # Initialize weights
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, x):
        """
        x: (batch_size, seq_len, num_features)
        Returns:
            price_mu: (B,)
            price_logvar: (B,)
            direction_logits: (B, 3)
            vol_pred: (B,)
        """
        B, T, _ = x.shape
        # Input projection + positional encoding
        pos = torch.arange(0, T, device=x.device).unsqueeze(0)  # (1, T)
        tok_emb = self.input_proj(x)  # (B, T, d_model)
        pos_emb = self.pos_embed(pos) # (1, T, d_model)
        h = tok_emb + pos_emb

        # Causal mask
        causal_mask = torch.tril(torch.ones(T, T, device=x.device)).bool()  # (T,T)

        for block in self.blocks:
            h = block(h, causal_mask)

        h = self.final_ln(h)
        # Take the representation of the last time step (most recent day)
        last_h = h[:, -1, :]  # (B, d_model)

        # Heads
        mu = self.price_mu(last_h).squeeze(-1)        # (B,)
        logvar = self.price_logvar(last_h).squeeze(-1)# (B,)
        dir_logits = self.direction_head(last_h)      # (B, 3)
        vol = self.volatility_head(last_h).squeeze(-1)# (B,)

        return mu, logvar, dir_logits, vol

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class MultiTaskLoss(nn.Module):
    def __init__(self, lambda_price=1.0, lambda_dir=0.5, lambda_vol=0.5):
        super().__init__()
        self.lambda_price = lambda_price
        self.lambda_dir = lambda_dir
        self.lambda_vol = lambda_vol
        self.mse = nn.MSELoss()

    def gaussian_nll(self, mu, logvar, target):
        """Negative log-likelihood of a Gaussian distribution."""
        precision = torch.exp(-logvar)
        loss = 0.5 * (logvar + (target - mu)**2 * precision)
        return loss.mean()

    def forward(self, price_mu, price_logvar, price_target,
                dir_logits, dir_target, vol_pred, vol_target):
        loss_price = self.gaussian_nll(price_mu, price_logvar, price_target)
        loss_dir = F.cross_entropy(dir_logits, dir_target)
        loss_vol = self.mse(vol_pred, vol_target)

        total = (self.lambda_price * loss_price +
                 self.lambda_dir * loss_dir +
                 self.lambda_vol * loss_vol)
        return total, {
            'loss_price': loss_price.item(),
            'loss_dir': loss_dir.item(),
            'loss_vol': loss_vol.item()
        }

In [ ]:
# Simulate batch: batch_size=4, seq_len=60, num_features=30 (example)
num_features = 30
model = StockTransformer(num_features=num_features, max_seq_len=60)
x = torch.randn(4, 60, num_features)
mu, logvar, dir_logits, vol = model(x)
print("Output shapes:")
print("Price mu:", mu.shape)
print("Price logvar:", logvar.shape)
print("Direction logits:", dir_logits.shape)
print("Volatility:", vol.shape)

# Test loss
loss_fn = MultiTaskLoss()
price_target = torch.randn(4)
dir_target = torch.randint(0, 3, (4,))
vol_target = torch.rand(4)
total_loss, losses_dict = loss_fn(mu, logvar, price_target,
                                      dir_logits, dir_target,
                                      vol, vol_target)
print(f"Total loss: {total_loss.item():.4f}")
print(f"Individual losses: {losses_dict}")

In [ ]:
import torch
from torchsummary import summary



# Define model parameters based on the pre-training configuration in cell t5XAxFUARn0_
NUM_FEATURES = 37 # As derived from PreTrainingDataset in t5XAxFUARn0_
D_MODEL = 256
N_HEADS = 8
N_LAYERS = 6
D_FF = 512
MAX_SEQ_LEN = 252
DROPOUT = 0.1
SEQ_LEN = 60 # From pre-training config

# Instantiate the model
model = StockTransformer(
    num_features=NUM_FEATURES,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_layers=N_LAYERS,
    d_ff=D_FF,
    max_seq_len=MAX_SEQ_LEN,
    dropout=DROPOUT
)

# Print model summary
print("\n--- StockTransformer Model Summary ---")
# The input size should be (seq_len, num_features) excluding batch dimension
summary(model, input_size=(SEQ_LEN, NUM_FEATURES), device='cpu')


### Pre‑Training Loop (Multi‑Stock, Price Head Only)

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from pathlib import Path
from tqdm import tqdm
import numpy as np

#training and model configs
PROCESSED_DIR = Path("data/processed")
SEQ_LEN = 60
BATCH_SIZE = 128
D_MODEL = 256
N_HEADS = 8
N_LAYERS = 6
D_FF = 512
MAX_SEQ_LEN = 252
DROPOUT = 0.1
LR = 3e-4
EPOCHS = 3
VAL_SPLIT = 0.05
CHECKPOINT_DIR = Path("checkpoints")
CHECKPOINT_DIR.mkdir(exist_ok=True)


# 1. Prepare dataset and dataloaders

# Load all ticker files
ticker_files = list(PROCESSED_DIR.glob("*.parquet"))
print(f"Found {len(ticker_files)} processed ticker files.")

# Create full dataset
full_dataset = PreTrainingDataset(
    data_dir=str(PROCESSED_DIR),
    tickers=None,           # use all
    seq_len=SEQ_LEN,
    feature_cols=None       # use all numeric columns
)
num_features = full_dataset.get_feature_dim()
print(f"Feature dimension: {num_features}")

# Train / validation split
val_size = int(len(full_dataset) * VAL_SPLIT)
train_size = len(full_dataset) - val_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size],
                                          generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# -------------------------------
# 2. Model, Optimizer, Scheduler
# -------------------------------
model = StockTransformer(num_features=num_features, d_model=D_MODEL,
                         n_heads=N_HEADS, n_layers=N_LAYERS,
                         d_ff=D_FF, max_seq_len=MAX_SEQ_LEN, dropout=DROPOUT)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

# Loss: for pre-training we only use the price head (Gaussian NLL)
# We'll reuse the MultiTaskLoss but only the price part; or define a simple Gaussian NLL function
def gaussian_nll_loss(mu, logvar, target):
    logvar = torch.clamp(logvar, min=-10, max=5)
    precision = torch.exp(-logvar)
    loss = 0.5 * (logvar + (target - mu)**2 * precision)
    return loss.mean()

# -------------------------------
# 3. Training Loop
# -------------------------------
best_val_loss = float('inf')

for epoch in range(EPOCHS):
    model.train()
    train_losses = []
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
    for batch in pbar:
        x, y_log_ret, _ = batch  # x: (B, T, F), y_price: (B, T), mask: (B, T)
        x = x.to(device)
        y_log_ret = y_log_ret.to(device)
        # We need the target at the LAST position only (masked)
        # Extract the last valid price target

        optimizer.zero_grad()
        mu, logvar, _, _ = model(x)  # ignore direction and volatility outputs

        loss = gaussian_nll_loss(mu, logvar, y_log_ret)
        loss.backward()

        # --- Gradient monitoring ---
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        if grad_norm.isnan():
            print("WARNING: Gradient norm is NaN!")
        elif grad_norm.isinf():
            print("WARNING: Gradient norm is Inf!")
        # Optional: log grad_norm more frequently or save to a list for analysis
        # pbar.set_postfix(loss=f"{loss.item():.4f}", grad_norm=f"{grad_norm:.2f}")
        # -------------------------

        optimizer.step()

        train_losses.append(loss.item())
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    avg_train_loss = np.mean(train_losses)

    # Validation
    model.eval()
    val_losses = []
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]"):
            x, y_log_ret,_ = batch
            x = x.to(device)
            y_log_ret = y_log_ret.to(device)

            mu, logvar, _, _ = model(x)
            loss = gaussian_nll_loss(mu, logvar, y_log_ret)
            val_losses.append(loss.item())

    avg_val_loss = np.mean(val_losses)
    print(f"Epoch {epoch+1}: Train Loss = {avg_train_loss:.4f}, Val Loss = {avg_val_loss:.4f}")

    # Checkpointing
    checkpoint = {
        'epoch': epoch + 1,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'val_loss': avg_val_loss,
        'num_features': num_features,
        'config': {
            'd_model': D_MODEL,
            'n_heads': N_HEADS,
            'n_layers': N_LAYERS,
            'seq_len': SEQ_LEN
        }
    }
    torch.save(checkpoint, CHECKPOINT_DIR / 'last.pth')
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(checkpoint, CHECKPOINT_DIR / 'best.pth')
        print(" -> Saved new best model.")

scheduler.step()

print("Pre-training finished.")

### LoRA Layer and Injection for the Transformer

In [ ]:
import torch
import torch.nn as nn
import math

class LoRALayer(nn.Module):
    """
    Applies Low-Rank Adaptation (LoRA) to a given linear layer.
    y = Wx + (alpha/r) * (B A x)   (A and B are the learned low-rank matrices)
    """
    def __init__(self, original_linear: nn.Linear, r: int = 8, alpha: int = 16, dropout: float = 0.05):
        super().__init__()
        self.original_linear = original_linear
        self.r = r
        self.alpha = alpha
        self.scaling = alpha / r

        # Freeze the original weights
        self.original_linear.weight.requires_grad = False
        if self.original_linear.bias is not None:
            self.original_linear.bias.requires_grad = False

        in_features = original_linear.in_features
        out_features = original_linear.out_features

        # Low-rank matrices
        self.lora_A = nn.Parameter(torch.zeros(in_features, r))
        self.lora_B = nn.Parameter(torch.zeros(r, out_features))
        self.lora_dropout = nn.Dropout(dropout)

        # Initialization: A normal, B zero so that initially delta is zero
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)

    def forward(self, x):
        # Original frozen output
        frozen_out = self.original_linear(x)
        # LoRA path
        lora_out = (self.lora_dropout(x) @ self.lora_A) @ self.lora_B
        return frozen_out + lora_out * self.scaling


def inject_lora_to_transformer(
    model: nn.Module,
    target_modules: list = ["W_q", "W_k", "W_v", "W_o"],
    r: int = 8,
    alpha: int = 16,
    dropout: float = 0.05
):
    """
    Replace specified linear layers in the StockTransformer with LoRALayer wrappers.
    The model must have attributes named like blocks[i].attn.W_q etc.
    """
    for name, module in model.named_modules():
        # We only wrap specific target modules inside MultiHeadCausalAttention
        if name.split('.')[-1] in target_modules and isinstance(module, nn.Linear):
            # Get the parent module (the attention block)
            parent_name = '.'.join(name.split('.')[:-1])
            parent = dict(model.named_modules())[parent_name]
            # Replace the module with LoRALayer
            setattr(parent, name.split('.')[-1], LoRALayer(module, r, alpha, dropout))
    print(f"Injected LoRA (r={r}, alpha={alpha}) into {target_modules} modules.")
    # Count trainable parameters
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

###  Fine‑Tuning Loop (Single Stock, Multi‑Head with Optional LoRA)

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from pathlib import Path
from tqdm import tqdm
import numpy as np
import pandas as pd

# -------------------------------
# Configuration
# -------------------------------
FINE_TUNE_TICKER = "AAPL.US"          # Choose the stock
SEQ_LEN = 60
BATCH_SIZE = 32
LR_FT = 1e-4
EPOCHS_FT = 50
CHECKPOINT_DIR = Path("checkpoints")
PRETRAINED_PATH = CHECKPOINT_DIR / "best.pth"   # or "last.pth"
PROCESSED_DIR = Path("data/processed")

# LoRA settings (optional)
USE_LORA = False   # Set True to inject LoRA adapters into linear layers
LORA_RANK = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

# -------------------------------
# 1. Prepare Single-Stock Dataset (first to determine num_features for model)
# -------------------------------
# Load the processed features for the chosen ticker
ticker_path = PROCESSED_DIR / f"{FINE_TUNE_TICKER}.parquet"
df = pd.read_parquet(ticker_path)

# Correct feature columns: exclude only the primary target column
# The pre-trained model was trained with 36 features, implying only 'Target_Log_Ret_Next' was excluded.
feature_cols = [c for c in df.columns if c not in ["Target_Log_Ret_Next"]]
num_features_ft = len(feature_cols) # This should now be 36

# Split into train/val (time-series split: last 10% as validation)
split_idx = int(len(df) * 0.9)
train_df = df.iloc[:split_idx]
val_df = df.iloc[split_idx:]

train_dataset = FineTuneDataset(train_df, seq_len=SEQ_LEN, feature_cols=feature_cols,
                                target_price_col="Target_Log_Ret_Next",
                                target_direction_col="Target_Direction",
                                target_vol_col="Target_Vol_Next")

# Pass the scaler from the training dataset to the validation dataset
val_dataset = FineTuneDataset(val_df, seq_len=SEQ_LEN, feature_cols=feature_cols,
                              target_price_col="Target_Log_Ret_Next",
                              target_direction_col="Target_Direction",
                              target_vol_col="Target_Vol_Next",
                              scaler=train_dataset.scaler)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}")
print(f"Number of features for fine-tuning: {num_features_ft}")

# -------------------------------
# 2. Load Pre-trained Model
# -------------------------------
# Fix: Set weights_only=False to allow loading of the checkpoint containing numpy objects.
# This is safe as the checkpoint was generated within this trusted notebook environment.
checkpoint = torch.load(PRETRAINED_PATH, map_location="cuda", weights_only=False)
model = StockTransformer(
    num_features=num_features_ft, # Use the dynamically determined number of features for fine-tuning
    d_model=checkpoint["config"]["d_model"],
    n_heads=checkpoint["config"]["n_heads"],
    n_layers=checkpoint["config"]["n_layers"],
    d_ff=512,   # assume fixed or store in config
    max_seq_len=252,
    dropout=0.1
)
model.load_state_dict(checkpoint["model_state_dict"])
print("Pre-trained model loaded.")

# -------------------------------
# 3. Inject LoRA (if enabled)
# -------------------------------
if USE_LORA:
    # Make sure Cell 9a (LoRA classes) has been executed
    inject_lora_to_transformer(
        model,
        target_modules=["W_q", "W_k", "W_v", "W_o"],  # Adapt attention projections
        r=LORA_RANK,
        alpha=LORA_ALPHA,
        dropout=LORA_DROPOUT
    )
    # LoRA layers already handle freezing of original weights,
    # but we also need to freeze all other parameters (like input_proj, heads, etc.)
    # Actually, inject_lora_to_transformer only freezes the wrapped linear layers.
    # We'll freeze the rest manually to ensure only LoRA params + heads are trained.
    for name, param in model.named_parameters():
        if 'lora_' not in name:  # LoRA parameters are called lora_A, lora_B
            param.requires_grad = False
    # Unfreeze the output heads (price, direction, volatility) so they can adapt
    for head_name in ['price_mu', 'price_logvar', 'direction_head', 'volatility_head']:
        for param in getattr(model, head_name).parameters():
            param.requires_grad = True
    # Also unfreeze final layer norm (optional, often helps)
    for param in model.final_ln.parameters():
        param.requires_grad = True
    # Print trainable parameters summary
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"LoRA enabled: {trainable:,} / {total:,} trainable parameters ({100*trainable/total:.2f}%%)")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


# -------------------------------
# 4. Training Setup
# -------------------------------
optimizer = torch.optim.AdamW(model.parameters(), lr=LR_FT)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_FT)
loss_fn = MultiTaskLoss(lambda_price=1.0, lambda_dir=0.5, lambda_vol=0.5)

best_val_loss = float('inf')

# -------------------------------
# 5. Fine-Tuning Loop
# -------------------------------
for epoch in range(EPOCHS_FT):
    model.train()
    train_losses = []
    pbar = tqdm(train_loader, desc=f"Fine-tune Epoch {epoch+1}/{EPOCHS_FT} [Train]")
    for batch in pbar:
        x, price_target, dir_target, vol_target = [t.to(device) for t in batch]

        optimizer.zero_grad()
        mu, logvar, dir_logits, vol_pred = model(x)
        total_loss, losses_dict = loss_fn(mu, logvar, price_target,
                                          dir_logits, dir_target,
                                          vol_pred, vol_target)
        total_loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        train_losses.append(total_loss.item())
        pbar.set_postfix(loss=f"{total_loss.item():.4f}", price=f"{losses_dict['loss_price']:.4f}",
                         dir=f"{losses_dict['loss_dir']:.4f}", vol=f"{losses_dict['loss_vol']:.4f}")

    avg_train_loss = np.mean(train_losses)

    # Validation
    model.eval()
    val_losses = []
    with torch.no_grad():
        for batch in val_loader:
            x, price_target, dir_target, vol_target = [t.to(device) for t in batch]
            mu, logvar, dir_logits, vol_pred = model(x)
            total_loss, _ = loss_fn(mu, logvar, price_target,
                                   dir_logits, dir_target,
                                   vol_pred, vol_target)
            val_losses.append(total_loss.item())
    avg_val_loss = np.mean(val_losses)
    print(f"Epoch {epoch+1}: Train Loss = {avg_train_loss:.4f}, Val Loss = {avg_val_loss:.4f}")

    # Save best fine-tuned model
    checkpoint_ft = {
        'epoch': epoch+1,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'val_loss': avg_val_loss,
        'num_features': num_features_ft, # Save the correct num_features for fine-tuning
        'config': checkpoint["config"] # Add config dictionary
    }
    torch.save(checkpoint_ft, CHECKPOINT_DIR / f'finetuned_{FINE_TUNE_TICKER}.pth')
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(checkpoint_ft, CHECKPOINT_DIR / 'best.pth')
        print(" -> Saved new best fine-tuned model.")

    scheduler.step()

print("Fine-tuning completed.")

### Inference

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

# -------------------------------
# 1. Load Model and Checkpoint
# -------------------------------
CHECKPOINT_DIR = Path("checkpoints")
FINE_TUNE_TICKER = "AAPL.US"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load fine‑tuned checkpoint (if exists), else fall back to pre‑trained best
ft_path = CHECKPOINT_DIR / f"finetuned_{FINE_TUNE_TICKER}.pth"
pretrain_path = CHECKPOINT_DIR / "best.pth"
if ft_path.exists():
    checkpoint_path = ft_path
    print("Loading fine‑tuned model...")
else:
    checkpoint_path = pretrain_path
    print("Loading pre‑trained model (no fine‑tuned found)...")

checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)

# We need num_features from the checkpoint; if missing, we'll infer from processed data
num_features = checkpoint.get("num_features", None)
if num_features is None:
    raise ValueError("num_features not found in checkpoint. Please re‑save with it.")

model = StockTransformer(
    num_features=num_features,
    d_model=checkpoint["config"]["d_model"],
    n_heads=checkpoint["config"]["n_heads"],
    n_layers=checkpoint["config"]["n_layers"],
    d_ff=512,   # fixed from original training
    max_seq_len=252,
    dropout=0.1
)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(DEVICE)
model.eval()
print("Model ready for inference.")

# -------------------------------
# 2. Prepare Data (single stock)
# -------------------------------
PROCESSED_DIR = Path("data/processed")
df = pd.read_parquet(PROCESSED_DIR / f"{FINE_TUNE_TICKER}.parquet")

# Fix: Align feature column selection with pre-training and fine-tuning datasets.
# The model was trained with num_features=36, by excluding only 'Target_Log_Ret_Next'.
# Reverting to the logic that produced 36 features for consistency.
feature_cols = [c for c in df.columns if c != "Target_Log_Ret_Next"]

# Identify index of raw 'Close' in the original dataframe (for conversion)
# Note: 'Close' is now part of feature_cols, so we must be careful not to use it as a target.
# However, it's needed for converting log-returns back to prices.
# We can still get its index if needed, but it should not be treated as a target here.
# close_idx = df.columns.get_loc("Close") # No longer needed this way, Close is a feature

SEQ_LEN = 60   # must match training sequence length

# Time‑series split (same as training: last 10% for test)
split_idx = int(len(df) * 0.9)
train_df = df.iloc[:split_idx]
test_df = df.iloc[split_idx:]

# Extract arrays
# Explicitly ensure float32 for all feature data
train_features_arr = train_df[feature_cols].values.astype(np.float32)
test_features = test_df[feature_cols].values.astype(np.float32)  # model inputs
test_close_raw = test_df["Close"].values                         # true close prices (for log‑return → price)
test_target_close = test_df["Target_Close_Next"].values          # actual next day close (ground truth)
test_target_dir = test_df["Target_Direction"].values
test_target_vol = test_df["Target_Vol_Next"].values

print(f"Test samples: {len(test_features)} (features), {len(test_target_close)} (targets)")

# -------------------------------
# 3. Single‑Step Prediction over Test Set
# -------------------------------
all_predicted_prices = []      # price = last_close * exp(mu)
all_predicted_sigmas = []      # uncertainty
all_dir_probs = []
all_vol_preds = []

with torch.no_grad():
    for i in range(len(test_features)):
        # Build input window of length SEQ_LEN ending at position i
        # If we don't have enough test data at the start, prepad with training data
        if i + 1 < SEQ_LEN:
            need = SEQ_LEN - (i + 1)
            # Ensure `train_features_arr` is used and cast to float32
            x_input = np.vstack([
                train_features_arr[-need:],
                test_features[:i+1]
            ]).astype(np.float32) # Ensure final array is float32
        else:
            x_input = test_features[i - SEQ_LEN + 1 : i + 1].astype(np.float32) # Ensure final array is float32

        # Last known close price (from the day i in test set)
        last_close = test_close_raw[i]   # because the window ends at day i, and we predict for day i+1

        # Model forward pass
        x_tensor = torch.from_numpy(x_input).unsqueeze(0).to(DEVICE).float()  # (1, SEQ_LEN, F) and explicit float()
        mu_logret, logvar, dir_logits, vol_pred = model(x_tensor)

        # Convert log return to price prediction
        mu_logret = mu_logret.item()
        sigma_logret = torch.exp(0.5 * torch.clamp(logvar, min=-10, max=5)).item()
        predicted_price = last_close * np.exp(mu_logret)

        all_predicted_prices.append(predicted_price)
        all_predicted_sigmas.append(sigma_logret)   # this is log‑return sigma, we could also compute price sigma
        all_dir_probs.append(F.softmax(dir_logits, dim=-1).cpu().numpy().flatten())
        all_vol_preds.append(vol_pred.item())

all_predicted_prices = np.array(all_predicted_prices)
all_predicted_sigmas = np.array(all_predicted_sigmas)
all_dir_probs = np.array(all_dir_probs)
all_vol_preds = np.array(all_vol_preds)

# Align: we predicted for every day i (0 to len-1). The target for that prediction is the next day's close.
# So all_predicted_prices[i] should match test_target_close[i] (since test_target_close[i] is Target_Close_Next for day i).
# We'll just use the full arrays directly (they have same length if we looped to len(test_features)).
# Note: test_target_close[-1] corresponds to the last day's next close, which we cannot evaluate because we don't have the next day's features? Actually our loop went up to len(test_features)-1? Wait: we looped `for i in range(len(test_features))`. At i = len(test_features)-1, we built a window ending at the last day of test set, and predicted for the day after that. But we have no ground truth for that day. So we should stop at i = len(test_features)-1. So we'll adjust alignment by dropping the last prediction.

# Trim to match ground truth length: predictions for i=0 to N-2 (predicting for day 1 to N-1).
pred_prices_align = all_predicted_prices[:-1]
true_prices_align = test_target_close[:-1]
dir_probs_align = all_dir_probs[:-1]
true_dir_align = test_target_dir[:-1]
vol_preds_align = all_vol_preds[:-1]
true_vol_align = test_target_vol[:-1]

# -------------------------------
# 4. Evaluation Metrics
# -------------------------------
def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred)**2))
def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))
def mape(y_true, y_pred, eps=1e-8):
    return np.mean(np.abs((y_true - y_pred) / (y_true + eps))) * 100
def directional_accuracy(y_true_dir, y_pred_probs):
    pred_dir = np.argmax(y_pred_probs, axis=1)
    return np.mean(pred_dir == y_true_dir) * 100

print("\n--- Evaluation Metrics ---")
print(f"RMSE: {rmse(true_prices_align, pred_prices_align):.4f}")
print(f"MAE:  {mae(true_prices_align, pred_prices_align):.4f}")
print(f"MAPE: {mape(true_prices_align, pred_prices_align):.2f}%")
print(f"Directional Accuracy: {directional_accuracy(true_dir_align, dir_probs_align):.2f}%")

# -------------------------------
# 5. Visualisation of Predictions vs True
# -------------------------------
# Use test_df indices for alignment (first N-1 rows)
plot_dates = test_df.index[:-1]

plt.figure(figsize=(14,6))
plt.plot(plot_dates, true_prices_align, label="True Next Close", color="blue", lw=1)
plt.plot(plot_dates, pred_prices_align, label="Predicted Next Close", color="red", lw=1)
# Approximate price uncertainty: multiply close price by log-return sigma? Actually price ≈ last_close * (1 + mu) for small mu, so price sigma ≈ last_close * sigma. We'll plot ±2 * last_close * sigma for a rough band.
# For a more accurate price uncertainty we could use the delta method, but we'll approximate.
approx_price_sigma = test_close_raw[:-1] * all_predicted_sigmas[:-1]  # last_close * sigma_logret
plt.fill_between(plot_dates,
                 pred_prices_align - 2*approx_price_sigma,
                 pred_prices_align + 2*approx_price_sigma,
                 alpha=0.2, color="red", label="±2σ Price Uncertainty")
plt.title(f"{FINE_TUNE_TICKER} – Single‑Step Predictions with Estimated Price Uncertainty")
plt.legend()
plt.tight_layout()
plt.show()

# -------------------------------
# 6. Multi‑Step Autoregressive Forecasting
# -------------------------------
forecast_horizon = 21
# Initial context: last SEQ_LEN days of the training set
initial_features = train_df[feature_cols].values[-SEQ_LEN:].astype(np.float32)
initial_close = train_df["Close"].values[-1]  # last known close price

context = torch.from_numpy(initial_features).unsqueeze(0).to(DEVICE)
predicted_prices = []
predicted_sigmas = []

# Get the index of 'Close' inside feature_cols if it exists (but we excluded it). So we don't have Close in the context. Instead we will update a separate variable for the closing price and manually build the new feature vector. We need to generate the next day's features. Without recomputing technical indicators, we can only keep a few features constant (like the most recent values for volume, etc.) and update the raw OHLCV-like features. For a minimalist approach, we will hold all features constant except we will generate a synthetic feature vector using the predicted price. This is not fully accurate but demonstrates the autoregressive mechanism. In a production system, we would recompute indicators each step. Here, we'll just set the next day's features as a copy of the previous day's feature vector, but we need to update the price-related features. The simplest: we'll use the actual feature vector of the last day of training data and just overwrite the price-related ones with predicted price. However, the model uses many derived features (returns, volatility, etc.). To be clean, we'll construct a next-day feature vector by taking the previous day's vector and adjusting only the "Close" related components. Since we excluded raw Close from input, we can't directly. So we'll need to include a few basic price features that we can update manually. Perhaps we can create a set of "recomputable" features: we'll add raw Close back into the feature set? That could cause data leakage. Instead, we can keep raw Close out but track it separately for conversion. For generating the next input, we will need to produce a full feature vector. This is complex without a feature recomputation engine. For demonstration, we will use a simplified version where we only use the actual test set features for the first step and then feed the model's own output as the next day's "Close" while keeping all other features constant. This is a common compromise in time-series generation without full feature recalculation. We'll note this limitation.

# For the simplified autoregressive: after each prediction, we create a new feature vector by copying the last input vector and updating the 'Return_1d' and 'Log_Ret_1d' and maybe 'Close'? Since Close is excluded, we need to know the index of 'Return_1d' and 'Log_Ret_1d' in feature_cols. We'll find them.
return1d_idx = feature_cols.index("Return_1d") if "Return_1d" in feature_cols else None
logret_idx = feature_cols.index("Log_Ret_1d") if "Log_Ret_1d" in feature_cols else None

context = torch.from_numpy(initial_features).unsqueeze(0).to(DEVICE).float()
last_close = initial_close

predicted_prices_multi = []
uncertainty_multi = []

model.eval()
with torch.no_grad():
    for step in range(forecast_horizon):
        mu_logret, logvar, _, _ = model(context)
        logvar = torch.clamp(logvar, min=-10, max=5)
        mu = mu_logret.item()
        sigma = torch.exp(0.5 * logvar).item()

        next_price = last_close * np.exp(mu)
        predicted_prices_multi.append(next_price)
        uncertainty_multi.append(sigma * last_close)   # price uncertainty

        # Prepare next input: shift context left, and append a synthetic feature vector
        new_features = context[0, -1, :].clone()   # take last day's features
        # Update return features using the predicted log return
        if logret_idx is not None:
            new_features[logret_idx] = mu   # the predicted log return becomes the feature for next day
        if return1d_idx is not None:
            # simple return = exp(log_return) - 1
            new_features[return1d_idx] = np.exp(mu) - 1.0

        # Append new day, drop first day
        context = torch.cat([context[:, 1:, :], new_features.unsqueeze(0).unsqueeze(0)], dim=1)
        last_close = next_price

# Plot multi-step forecast
plt.figure(figsize=(14,5))
plt.plot(range(len(predicted_prices_multi)), predicted_prices_multi, marker='o', label="Autoregressive Forecast")
plt.fill_between(range(len(predicted_prices_multi)),
                 np.array(predicted_prices_multi) - 2*np.array(uncertainty_multi),
                 np.array(predicted_prices_multi) + 2*np.array(uncertainty_multi),
                 alpha=0.2, label="±2σ Uncertainty")
plt.title(f"{FINE_TUNE_TICKER} – {forecast_horizon}-Day Autoregressive Forecast")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

# -------------------------------
# 1. Load Model and Checkpoint
# -------------------------------
CHECKPOINT_DIR = Path("checkpoints")
FINE_TUNE_TICKER = "AAPL.US"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load fine‑tuned checkpoint (if exists), else fall back to pre‑trained best
ft_path = CHECKPOINT_DIR / f"finetuned_{FINE_TUNE_TICKER}.pth"
pretrain_path = CHECKPOINT_DIR / "best.pth"
if ft_path.exists():
    checkpoint_path = ft_path
    print("Loading fine‑tuned model...")
else:
    checkpoint_path = pretrain_path
    print("Loading pre‑trained model (no fine‑tuned found)...")

checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)

# We need num_features from the checkpoint; if missing, we'll infer from processed data
num_features = checkpoint.get("num_features", None)
if num_features is None:
    raise ValueError("num_features not found in checkpoint. Please re‑save with it.")

model = StockTransformer(
    num_features=num_features,
    d_model=checkpoint["config"]["d_model"],
    n_heads=checkpoint["config"]["n_heads"],
    n_layers=checkpoint["config"]["n_layers"],
    d_ff=512,   # fixed from original training
    max_seq_len=252,
    dropout=0.1
)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(DEVICE)
model.eval()
print("Model ready for inference.")

# -------------------------------
# 2. Prepare Data (single stock)
# -------------------------------
PROCESSED_DIR = Path("data/processed")
df = pd.read_parquet(PROCESSED_DIR / f"{FINE_TUNE_TICKER}.parquet")

# Fix: Align feature column selection with pre-training and fine-tuning datasets.
# The model was trained with num_features=36, by excluding only 'Target_Log_Ret_Next'.
# Reverting to the logic that produced 36 features for consistency.
feature_cols = [c for c in df.columns if c != "Target_Log_Ret_Next"]

# Identify index of raw 'Close' in the original dataframe (for conversion)
# Note: 'Close' is now part of feature_cols, so we must be careful not to use it as a target.
# However, it's needed for converting log-returns back to prices.
# We can still get its index if needed, but it should not be treated as a target here.
# close_idx = df.columns.get_loc("Close") # No longer needed this way, Close is a feature

SEQ_LEN = 60   # must match training sequence length

# Time‑series split (same as training: last 10% for test)
split_idx = int(len(df) * 0.9)
train_df = df.iloc[:split_idx]
test_df = df.iloc[split_idx:]

# Extract arrays
# Explicitly ensure float32 for all feature data
train_features_arr = train_df[feature_cols].values.astype(np.float32)
test_features = test_df[feature_cols].values.astype(np.float32)  # model inputs
test_close_raw = test_df["Close"].values                         # true close prices (for log‑return → price)
test_target_close = test_df["Target_Close_Next"].values          # actual next day close (ground truth)
test_target_dir = test_df["Target_Direction"].values
test_target_vol = test_df["Target_Vol_Next"].values

print(f"Test samples: {len(test_features)} (features), {len(test_target_close)} (targets)")

# -------------------------------
# 3. Single‑Step Prediction over Test Set
# -------------------------------
all_predicted_prices = []      # price = last_close * exp(mu)
all_predicted_sigmas = []      # uncertainty
all_dir_probs = []
all_vol_preds = []

with torch.no_grad():
    for i in range(len(test_features)):
        # Build input window of length SEQ_LEN ending at position i
        # If we don't have enough test data at the start, prepad with training data
        if i + 1 < SEQ_LEN:
            need = SEQ_LEN - (i + 1)
            # Ensure `train_features_arr` is used and cast to float32
            x_input = np.vstack([
                train_features_arr[-need:],
                test_features[:i+1]
            ]).astype(np.float32) # Ensure final array is float32
        else:
            x_input = test_features[i - SEQ_LEN + 1 : i + 1].astype(np.float32) # Ensure final array is float32

        # Last known close price (from the day i in test set)
        last_close = test_close_raw[i]   # because the window ends at day i, and we predict for day i+1

        # Model forward pass
        x_tensor = torch.from_numpy(x_input).unsqueeze(0).to(DEVICE).float()  # (1, SEQ_LEN, F) and explicit float()
        mu_logret, logvar, dir_logits, vol_pred = model(x_tensor)

        # Convert log return to price prediction
        mu_logret = mu_logret.item()
        sigma_logret = torch.exp(0.5 * torch.clamp(logvar, min=-10, max=5)).item()
        predicted_price = last_close * np.exp(mu_logret)

        all_predicted_prices.append(predicted_price)
        all_predicted_sigmas.append(sigma_logret)   # this is log‑return sigma, we could also compute price sigma
        all_dir_probs.append(F.softmax(dir_logits, dim=-1).cpu().numpy().flatten())
        all_vol_preds.append(vol_pred.item())

all_predicted_prices = np.array(all_predicted_prices)
all_predicted_sigmas = np.array(all_predicted_sigmas)
all_dir_probs = np.array(all_dir_probs)
all_vol_preds = np.array(all_vol_preds)

# Align: we predicted for every day i (0 to len-1). The target for that prediction is the next day's close.
# So all_predicted_prices[i] should match test_target_close[i] (since test_target_close[i] is Target_Close_Next for day i).
# We'll just use the full arrays directly (they have same length if we looped to len(test_features)).
# Note: test_target_close[-1] corresponds to the last day's next close, which we cannot evaluate because we don't have the next day's features? Actually our loop went up to len(test_features)-1? Wait: we looped `for i in range(len(test_features))`. At i = len(test_features)-1, we built a window ending at the last day of test set, and predicted for the day after that. But we have no ground truth for that day. So we should stop at i = len(test_features)-1. So we'll adjust alignment by dropping the last prediction.

# Trim to match ground truth length: predictions for i=0 to N-2 (predicting for day 1 to N-1).
pred_prices_align = all_predicted_prices[:-1]
true_prices_align = test_target_close[:-1]
dir_probs_align = all_dir_probs[:-1]
true_dir_align = test_target_dir[:-1]
vol_preds_align = all_vol_preds[:-1]
true_vol_align = test_target_vol[:-1]

# -------------------------------
# 4. Evaluation Metrics
# -------------------------------
def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred)**2))
def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))
def mape(y_true, y_pred, eps=1e-8):
    return np.mean(np.abs((y_true - y_pred) / (y_true + eps))) * 100
def directional_accuracy(y_true_dir, y_pred_probs):
    pred_dir = np.argmax(y_pred_probs, axis=1)
    return np.mean(pred_dir == y_true_dir) * 100

print("\n--- Evaluation Metrics ---")
print(f"RMSE: {rmse(true_prices_align, pred_prices_align):.4f}")
print(f"MAE:  {mae(true_prices_align, pred_prices_align):.4f}")
print(f"MAPE: {mape(true_prices_align, pred_prices_align):.2f}%")
print(f"Directional Accuracy: {directional_accuracy(true_dir_align, dir_probs_align):.2f}%")

# -------------------------------
# 5. Visualisation of Predictions vs True
# -------------------------------
# Use test_df indices for alignment (first N-1 rows)
plot_dates = test_df.index[:-1]

plt.figure(figsize=(14,6))
plt.plot(plot_dates, true_prices_align, label="True Next Close", color="blue", lw=1)
plt.plot(plot_dates, pred_prices_align, label="Predicted Next Close", color="red", lw=1)
# Approximate price uncertainty: multiply close price by log-return sigma? Actually price ≈ last_close * (1 + mu) for small mu, so price sigma ≈ last_close * sigma. We'll plot ±2 * last_close * sigma for a rough band.
# For a more accurate price uncertainty we could use the delta method, but we'll approximate.
approx_price_sigma = test_close_raw[:-1] * all_predicted_sigmas[:-1]  # last_close * sigma_logret
plt.fill_between(plot_dates,
                 pred_prices_align - 2*approx_price_sigma,
                 pred_prices_align + 2*approx_price_sigma,
                 alpha=0.2, color="red", label="±2σ Price Uncertainty")
plt.title(f"{FINE_TUNE_TICKER} – Single‑Step Predictions with Estimated Price Uncertainty")
plt.legend()
plt.tight_layout()
plt.show()

# -------------------------------
# 6. Multi‑Step Autoregressive Forecasting
# -------------------------------
forecast_horizon = 21
# Initial context: last SEQ_LEN days of the training set
initial_features = train_df[feature_cols].values[-SEQ_LEN:].astype(np.float32)
initial_close = train_df["Close"].values[-1]  # last known close price

context = torch.from_numpy(initial_features).unsqueeze(0).to(DEVICE)
predicted_prices = []
predicted_sigmas = []

# Get the index of 'Close' inside feature_cols if it exists (but we excluded it). So we don't have Close in the context. Instead we will update a separate variable for the closing price and manually build the new feature vector. We need to generate the next day's features. Without recomputing technical indicators, we can only keep a few features constant (like the most recent values for volume, etc.) and update the raw OHLCV-like features. For a minimalist approach, we will hold all features constant except we will generate a synthetic feature vector using the predicted price. This is not fully accurate but demonstrates the autoregressive mechanism. In a production system, we would recompute indicators each step. Here, we'll just set the next day's features as a copy of the previous day's feature vector, but we need to update the price-related features. The simplest: we'll use the actual feature vector of the last day of training data and just overwrite the price-related ones with predicted price. However, the model uses many derived features (returns, volatility, etc.). To be clean, we'll construct a next-day feature vector by taking the previous day's vector and adjusting only the "Close" related components. Since we excluded raw Close from input, we can't directly. So we'll need to include a few basic price features that we can update manually. Perhaps we can create a set of "recomputable" features: we'll add raw Close back into the feature set? That could cause data leakage. Instead, we can keep raw Close out but track it separately for conversion. For generating the next input, we will need to produce a full feature vector. This is complex without a feature recomputation engine. For demonstration, we will use a simplified version where we only use the actual test set features for the first step and then feed the model's own output as the next day's "Close" while keeping all other features constant. This is a common compromise in time-series generation without full feature recalculation. We'll note this limitation.

# For the simplified autoregressive: after each prediction, we create a new feature vector by copying the last input vector and updating the 'Return_1d' and 'Log_Ret_1d' and maybe 'Close'? Since Close is excluded, we need to know the index of 'Return_1d' and 'Log_Ret_1d' in feature_cols. We'll find them.
return1d_idx = feature_cols.index("Return_1d") if "Return_1d" in feature_cols else None
logret_idx = feature_cols.index("Log_Ret_1d") if "Log_Ret_1d" in feature_cols else None

context = torch.from_numpy(initial_features).unsqueeze(0).to(DEVICE).float()
last_close = initial_close

predicted_prices_multi = []
uncertainty_multi = []

model.eval()
with torch.no_grad():
    for step in range(forecast_horizon):
        mu_logret, logvar, _, _ = model(context)
        logvar = torch.clamp(logvar, min=-10, max=5)
        mu = mu_logret.item()
        sigma = torch.exp(0.5 * logvar).item()

        next_price = last_close * np.exp(mu)
        predicted_prices_multi.append(next_price)
        uncertainty_multi.append(sigma * last_close)   # price uncertainty

        # Prepare next input: shift context left, and append a synthetic feature vector
        new_features = context[0, -1, :].clone()   # take last day's features
        # Update return features using the predicted log return
        if logret_idx is not None:
            new_features[logret_idx] = mu   # the predicted log return becomes the feature for next day
        if return1d_idx is not None:
            # simple return = exp(log_return) - 1
            new_features[return1d_idx] = np.exp(mu) - 1.0

        # Append new day, drop first day
        context = torch.cat([context[:, 1:, :], new_features.unsqueeze(0).unsqueeze(0)], dim=1)
        last_close = next_price

# Plot multi-step forecast
plt.figure(figsize=(14,5))
plt.plot(range(len(predicted_prices_multi)), predicted_prices_multi, marker='o', label="Autoregressive Forecast")
plt.fill_between(range(len(predicted_prices_multi)),
                 np.array(predicted_prices_multi) - 2*np.array(uncertainty_multi),
                 np.array(predicted_prices_multi) + 2*np.array(uncertainty_multi),
                 alpha=0.2, label="±2σ Uncertainty")
plt.title(f"{FINE_TUNE_TICKER} – {forecast_horizon}-Day Autoregressive Forecast")
plt.legend()
plt.tight_layout()
plt.show()


### Random Forest Baseline for Directional Accuracy

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Reuse the dataframes and scaler from the Transformer setup
# `train_df`, `test_df`, `feature_cols` and `train_dataset.scaler` are available from previous cells.

# Fix for data leakage: Create a new feature_cols list for Random Forest
# that explicitly excludes the target columns and any other potentially leaking features.
# The original `feature_cols` from the Transformer setup (H2nEapajS1eJ) had 'Target_Close_Next', 'Target_Direction', 'Target_Vol_Next' implicitly as features due to the broad definition.
# We need to ensure the Random Forest does NOT see these.
rf_feature_cols = [c for c in feature_cols if c not in ["Target_Close_Next", "Target_Direction", "Target_Vol_Next"]]

# Extract features and target for Random Forest
X_train_rf = train_df[rf_feature_cols].values
y_train_rf = train_df["Target_Direction"].values

X_test_rf = test_df[rf_feature_cols].values
y_test_rf = test_df["Target_Direction"].values

# Apply the same scaler used for the Transformer data
rfc_scaler = train_dataset.scaler
X_train_scaled_rf = rfc_scaler.transform(X_train_rf)
X_test_scaled_rf = rfc_scaler.transform(X_test_rf)

# Initialize and train Random Forest Classifier
print("Training Random Forest Classifier...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1) # n_jobs=-1 uses all available cores
rf_model.fit(X_train_scaled_rf, y_train_rf)
print("Random Forest training complete.")

# Predict probabilities on the test set
rf_pred_probs = rf_model.predict_proba(X_test_scaled_rf)

# Adjust `rf_pred_probs` to match the length of `true_dir_align`
# The FineTuneDataset drops the last `seq_len` samples for input windows,
# and the inference loop also trims the last prediction. So we need to ensure
# the RF predictions are aligned with `true_dir_align`.
# The `y_test_rf` already corresponds to the original `test_df` targets, so we trim it.
rf_pred_probs_align = rf_pred_probs[:-1] # Match the trimming of `true_dir_align`
rf_true_dir_align = y_test_rf[:-1] # Match the trimming of `true_dir_align`

# Calculate directional accuracy for Random Forest
rf_directional_accuracy = directional_accuracy(rf_true_dir_align, rf_pred_probs_align)

print(f"\n--- Directional Accuracy Comparison ---")
print(f"Transformer Directional Accuracy: {directional_accuracy(true_dir_align, dir_probs_align):.2f}%")
print(f"Random Forest Directional Accuracy: {rf_directional_accuracy:.2f}%")